In [1]:
# Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [115]:
import pandas as pd
#import the stance and weights

df = pd.read_csv("/content/drive/MyDrive/HSBC_720_with_stance.csv")

age_w = pd.read_csv("/content/drive/MyDrive/age_weights.csv")
region_w = pd.read_csv("/content/drive/MyDrive/region18_weights.csv")
income_w = pd.read_csv("/content/drive/MyDrive/income_weights.csv")
household_w = pd.read_csv("/content/drive/MyDrive/household_weights_norm.csv")
tech_w = pd.read_csv("/content/drive/MyDrive/tech_weights_norm.csv")

In [116]:
# 3. To dictionary
age_dict = dict(zip(age_w["Category"], age_w["Weight"]))
region_dict = dict(zip(region_w["Category"], region_w["Weight"]))
income_dict = dict(zip(income_w["Category"], income_w["Weight"]))
household_dict = dict(zip(household_w["Category"], household_w["Weight"]))
tech_dict = dict(zip(tech_w["Category"], tech_w["Weight"]))

In [117]:
# 4. match the weights
df["w_age"] = df["age_group"].map(age_dict)
df["w_region"] = df["region"].map(region_dict)
df["w_income"] = df["income"].map(income_dict)
df["w_household"] = df["household_composition"].map(household_dict)
df["w_tech"] = df["tech_adoption"].map(tech_dict)

In [118]:
# 5. check the match
print(df[["w_age", "w_region", "w_income", "w_household", "w_tech"]].isna().sum())

w_age          0
w_region       0
w_income       0
w_household    0
w_tech         0
dtype: int64


In [119]:
df[["w_age", "w_region", "w_income", "w_household", "w_tech"]].describe()

,w_age,w_region,w_income,w_household,w_tech
count,720.000000,720.000000,7.200000e+02,720.000000,720.000000
mean,0.200000,0.250000,3.333333e-01,0.250000,0.333333
std,0.072289,0.342872,3.388534e-15,0.083846,0.164912
min,0.104949,0.027660,3.333333e-01,0.142400,0.102300
25%,0.163105,0.042098,3.333333e-01,0.181250,0.102300
50%,0.169282,0.064939,3.333333e-01,0.260700,0.422300
75%,0.253740,0.272841,3.333333e-01,0.329450,0.475400
max,0.308924,0.842462,3.333333e-01,0.336200,0.475400


In [120]:
# 6. calculate original union weights
df["raw_weight"] = (
    df["w_age"] *
    df["w_region"] *
    df["w_income"] *
    df["w_household"] *
    df["w_tech"]
)

In [121]:
df["raw_weight"].describe()

,raw_weight
count,720.000000
mean,0.001389
std,0.002602
min,0.000014
25%,0.000134
50%,0.000324
75%,0.000989
max,0.013866


In [122]:
# 7. Normalise
df["final_weight"] = df["raw_weight"] / df["raw_weight"].sum()

In [123]:
df["final_weight"].sum()

np.float64(0.9999999999999999)

In [124]:
# 8.  promoter / detractor / passive
df["predicted_stance_clean"] = df["predicted_stance"].str.strip().str.lower()

promoter_share = df.loc[
    df["predicted_stance_clean"] == "promoter",
    "final_weight"
].sum()

passive_share = df.loc[
    df["predicted_stance_clean"] == "passive",
    "final_weight"
].sum()

detractor_share = df.loc[
    df["predicted_stance_clean"] == "detractor",
    "final_weight"
].sum()

# 9. weighted NPS
weighted_nps = (promoter_share - detractor_share) * 100

print("Weighted Promoter Share:", round(promoter_share * 100, 2), "%")
print("Weighted Passive Share:", round(passive_share * 100, 2), "%")
print("Weighted Detractor Share:", round(detractor_share * 100, 2), "%")
print("Weighted NPS:", round(weighted_nps, 2))

Weighted Promoter Share: 19.5 %
Weighted Passive Share: 44.21 %
Weighted Detractor Share: 36.3 %
Weighted NPS: -16.8


In [ ]:
# Save
df.to_csv("FD_50_with_weighted_stance.csv", index=False)

In [24]:
# Calculate unweighted NPS
unweighted_promoter = (df["predicted_stance"].str.lower() == "promoter").mean()
unweighted_detractor = (df["predicted_stance"].str.lower() == "detractor").mean()

unweighted_nps = (unweighted_promoter - unweighted_detractor) * 100

print("Unweighted NPS:", unweighted_nps)

Unweighted NPS: 25.416666666666664
